In [0]:
DESCRIBE samples.nyctaxi.trips;

In [0]:
-- Ejercicio 0.1: CTE básica - Estadísticas por zona
WITH estadisticas_por_zona (
    SELECT 
        pickup_zip,
        COUNT(*) AS total_viajes,
        ROUND(AVG(fare_amount), 2) AS tarifa_promedio,
        ROUND(AVG(trip_distance), 2) AS distancia_promedio
    FROM samples.nyctaxi.trips
    WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
    GROUP BY pickup_zip
    HAVING total_viajes > 100
)

SELECT 
  pickup_zip,
  total_viajes,
  tarifa_promedio,
  distancia_promedio
FROM estadisticas_por_zona
ORDER BY total_viajes DESC
LIMIT 20;


In [0]:
-- Ejercicio 0.2: CTE múltiple - Análisis comparativo
WITH promedio_por_hora AS (
    SELECT 
    HOUR(tpep_pickup_datetime) AS hora_dia,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_hora
    FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
  GROUP BY HOUR(tpep_pickup_datetime)
),
promedio_general AS(
  SELECT 
    ROUND(AVG(fare_amount), 2) AS tarifa_promedio_total
  FROM samples.nyctaxi.trips
  WHERE fare_amount > 0
)
SELECT 
p.hora_dia,
p.cantidad_viajes,
p.tarifa_promedio_hora,
pg.tarifa_promedio_total,
ROUND(p.tarifa_promedio_hora - pg.tarifa_promedio_total, 2) AS diferencia,
ROUND((p.tarifa_promedio_hora - pg.tarifa_promedio_total) * 100.0 / pg.tarifa_promedio_total, 2) AS porcentaje_diferencia
FROM promedio_por_hora p
CROSS JOIN promedio_general pg
ORDER BY p.hora_dia ASC;

In [0]:
-- Ejercicio 0.3: CTE para limpieza de datos

WITH viajes_validos AS (
    SELECT 
    trip_distance, 
    fare_amount
    FROM samples.nyctaxi.trips 
    WHERE fare_amount > 0 
    AND trip_distance > 0
    AND trip_distance IS NOT NULL
    AND fare_amount IS NOT NULL
),

estadisticas AS (
    SELECT 
        COUNT(*) AS total_viajes_validos,
        ROUND(MIN(trip_distance), 2) AS minimo,
        ROUND(MAX(trip_distance), 2) AS maximo,
        ROUND(AVG(trip_distance), 2) AS promedio,
        ROUND(PERCENTILE(trip_distance, 0.5), 2) AS distancia_mediana
    FROM viajes_validos
)

SELECT * FROM estadisticas;




In [0]:
-- Ejercicio 0.4: Window Function - ROW_NUMBER() para ranking
SELECT 
ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS viajes_mas_caros,
tpep_pickup_datetime AS fecha,
ROUND(trip_distance, 2) AS distancia,
ROUND(fare_amount, 2) AS tarifa
FROM samples.nyctaxi.trips 
WHERE fare_amount > 0
ORDER BY fare_amount DESC
LIMIT 20; 


In [0]:
-- Ejercicio 0.5: Window Function - ROW_NUMBER() vs RANK() vs DENSE_RANK()

-- Pregunta: Usando la misma tabla de viajes, mostrá los top 20 viajes más caros con su fecha, tarifa y tres columnas de ranking: ROW_NUMBER(), RANK() y DENSE_RANK(). ¿Qué pasa cuando hay tarifas repetidas? Pista: Usá las 3 funciones con OVER (ORDER BY fare_amount DESC) en el mismo SELECT

SELECT 
fare_amount,
tpep_pickup_datetime AS fecha,
RANK() OVER (ORDER BY fare_amount DESC) AS rank,
ROW_NUMBER() OVER (ORDER BY fare_amount DESC) AS row_number,
DENSE_RANK() OVER (ORDER BY fare_amount DESC) AS dense_rank
FROM samples.nyctaxi.trips
LIMIT 20;


In [0]:
-- Ejercicio 0.6: Comparar con promedio general
SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa,
  ROUND(trip_distance, 2) AS distancia,
  ROUND(AVG(fare_amount) OVER(), 2) AS tarifa_promedio_general,
  ROUND(fare_amount - AVG(fare_amount) OVER(), 2) AS diferencia_con_promedio,
  ROUND((fare_amount - AVG(fare_amount) OVER()) * 100.0 / AVG(fare_amount) OVER(), 2) AS porcentaje_diferencia
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY fecha DESC
LIMIT 20;

In [0]:
-- Ejercicio 0.7: Promedio por partición
SELECT 
    pickup_zip AS zona,
    tpep_pickup_datetime AS fecha,
    ROUND(fare_amount, 2) AS tarifa,
    ROUND(trip_distance, 2) AS distancia,
    ROUND(AVG(fare_amount) OVER(PARTITION BY pickup_zip), 2) AS tarifa_promedio_por_zona,
    ROUND(fare_amount - AVG(fare_amount) OVER (PARTITION BY pickup_zip), 2) AS diferencia_con_zona
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
AND pickup_zip IS NOT NULL
ORDER BY pickup_zip, fare_amount DESC
LIMIT 20;

In [0]:
-- Ejercicio 0.8: LAG() para comparar con anterior
SELECT 
  tpep_pickup_datetime AS fecha,
  ROUND(fare_amount, 2) AS tarifa_actual,
  LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime) AS tarifa_anterior,
  ROUND(fare_amount - LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime), 2) AS diferencia_con_anterior
FROM samples.nyctaxi.trips
WHERE fare_amount > 0
ORDER BY tpep_pickup_datetime
LIMIT 50;


In [0]:
-- Ejercicio 0.9: SUM() acumulado
SELECT 
  tpep_pickup_datetime AS fecha,
  fare_amount AS tarifa_actual,
 SUM(fare_amount) OVER (ORDER BY tpep_pickup_datetime ROWS BETWEEN UNBOUNDED PRECEDING AND
CURRENT ROW) AS total_fare
FROM samples.nyctaxi.trips
WHERE trip_distance > 0
ORDER BY tpep_pickup_datetime;

In [0]:
-- Ejercicio 0.10: Combinando CTEs y Window Functions
-- Objetivo: Combinar ambas técnicas en una query compleja.
--Pregunta: Usa CTEs para:
--1. Filtrar viajes válidos (distancia > 0, tarifa > 0)
--2. Calcular estadísticas por zona (promedio, máximo, mínimo)
--3. Luego usa Window Functions para rankear las zonas por promedio de tarifa
--4. Muestra el top 10 zonas más caras con su ranking
--Pista: Combina WITH para las CTEs y ROW_NUMBER() OVER() para el ranking

WITH viajes_validos AS(
    SELECT 
        trip_distance,
        fare_amount,
        pickup_zip
    FROM samples.nyctaxi.trips
    WHERE pickup_zip IS NOT NULL
    AND fare_amount > 0
    AND trip_distance > 0
),
estadisticas_por_zona (
    SELECT 
        pickup_zip,
        COUNT(*) AS total_viajes,
        ROUND(AVG(fare_amount), 2) AS promedio_tarifa,
        ROUND(MIN(fare_amount), 2) AS minimo_tarifa,
        ROUND(MAX(fare_amount), 2) AS maximo_tarifa
    FROM viajes_validos
    GROUP BY pickup_zip
    HAVING COUNT(*) >= 50 -- Sólo zonas con más de 50 viajes
)
SELECT
    ROW_NUMBER() OVER (ORDER BY promedio_tarifa DESC) AS ranking,
    promedio_tarifa,
    minimo_tarifa,
    maximo_tarifa,
    pickup_zip
FROM estadisticas_por_zona
ORDER BY promedio_tarifa DESC
LIMIT 10;

